# Cost Model Comparison (grid:astar)

Compare all cost models using `grid:astar` across weights 1, 10, 100.
This notebook generates 20 elevation samples and visualizes resulting paths.

In [15]:
import json
import logging
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from loguru import logger as loguru_logger

# Configure info logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

# Configure loguru for info level
loguru_logger.remove()
loguru_logger.add(sys.stderr, level="INFO")

# Add src to path for imports
ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

In [16]:
from passage.generate import find_path, load_tile, _assemble_crop_from_tiles
from passage.pathfinding_utils import COST_MODELS
from scipy.spatial.distance import directed_hausdorff

In [17]:
# Parameters (papermill)
RESOLUTION = 256
NUM_SAMPLES = 1000
MAX_SAMPLES_DISPLAY = 0

In [18]:
# Configuration
CONFIG_PATH = ROOT / "config" / "config.yaml"
TILES_DIR = ROOT / "outputs" / "download" / "tiles"
CALIBRATE_PATH = ROOT / "outputs" / "calibrate" / "calibrate.json"

# WEIGHTS = [1e0, 1e1, 1e2, 1e3, 1e4, 1e5, 1e6, 1e7]
# WEIGHTS = [1e0, 1e1, 1e2, 1e3, 1e4, 1e5]
WEIGHTS = np.pow(10, np.linspace(0, 7, num=40)).tolist()
COST_MODELS_LIST = sorted(list(COST_MODELS))
TILE_SIZE_PX = 3600

print(f"Resolution: {RESOLUTION}")
print(f"Weights: {WEIGHTS}")
print(f"Cost models: {COST_MODELS_LIST}")

Resolution: 256
Weights: [1.0, 1.5117750706156623, 2.2854638641349903, 3.455107294592219, 5.223345074266842, 7.896522868499726, 11.937766417144363, 18.047217668271703, 27.28333376486768, 41.24626382901352, 62.35507341273915, 94.26684551178855, 142.5102670302998, 215.44346900318845, 325.7020655659783, 492.3882631706742, 744.3803013251688, 1125.3355826007646, 1701.2542798525892, 2571.913809059344, 3888.1551803080893, 5878.0160722749115, 8886.238162743408, 13433.993325989015, 20309.17620904735, 30702.9062975785, 46415.88833612782, 70170.38286703822, 106081.83551394482, 160371.8743751331, 242446.20170823307, 366524.1237079626, 554102.0330009492, 837677.6400682924, 1266380.173467402, 1914481.9761699575, 2894266.1247167517, 4375479.375074189, 6614740.6412301455, 10000000.0]
Cost models: ['distance', 'elevation', 'energy', 'slope_uphill']


In [19]:
# Load calibration data
if not CALIBRATE_PATH.exists():
    raise FileNotFoundError(f"Calibration file not found: {CALIBRATE_PATH}. Run 'passage calibrate' first.")
if not TILES_DIR.exists():
    raise FileNotFoundError(f"Tiles directory not found: {TILES_DIR}. Run 'passage download' first.")

with open(CALIBRATE_PATH) as f:
    calibration = json.load(f)

global_min = float(calibration["global_min"])
global_max = float(calibration["global_max"])
tile_stats_df = pd.DataFrame(calibration["tile_stats"])

print(f"Global elevation range: [{global_min:.1f}, {global_max:.1f}]")
print(f"Number of tiles: {len(tile_stats_df)}")

# outputs cleared

Global elevation range: [-430.0, 8767.0]
Number of tiles: 23993


In [20]:
import math
import random


def generate_random_crop(
    resolution: int,
    tile_stats_df: pd.DataFrame,
    tiles_dirpath: Path,
    global_min: float,
    global_max: float,
    seed: int | None = None,
) -> np.ndarray | None:
    if seed is not None:
        random.seed(seed)
        np.random.seed(seed)

    half_res = resolution // 2
    half_tiles = max(1, math.ceil(half_res / TILE_SIZE_PX))
    n_tiles = 2 * half_tiles + 1
    grid_h = n_tiles * TILE_SIZE_PX
    grid_w = n_tiles * TILE_SIZE_PX

    for _ in range(10):
        target_elev = random.uniform(global_min, global_max)
        candidates = tile_stats_df[(tile_stats_df["min"] <= target_elev) & (tile_stats_df["max"] >= target_elev)]
        if candidates.empty:
            continue
        selected = candidates.sample(n=1).iloc[0]
        center_lat = int(selected["lat"])
        center_lon = int(selected["lon"])
        if grid_h < resolution or grid_w < resolution:
            continue
        y1 = random.randint(0, grid_h - resolution)
        x1 = random.randint(0, grid_w - resolution)
        center_tile = load_tile(tiles_dirpath, center_lat, center_lon)
        crop = _assemble_crop_from_tiles(
            tiles_dirpath=tiles_dirpath,
            center_lat=center_lat,
            center_lon=center_lon,
            half_tiles=half_tiles,
            y1=y1,
            x1=x1,
            resolution=resolution,
            center_tile=center_tile,
        )
        return crop
    return None


def get_marker_positions(resolution: int) -> tuple[tuple[int, int], tuple[int, int]]:
    pos1 = int(0.2 * resolution)
    pos2 = int(0.8 * resolution)
    return (pos1, pos1), (pos2, pos2)

In [21]:
def compute_paths_for_crop(
    crop: np.ndarray, start: tuple[int, int], end: tuple[int, int], cost_models: list[str], weights: list[float]
) -> dict:
    results = {}
    for cost_model_name in cost_models:
        results[cost_model_name] = {}
        for w in weights:
            solver_config = {
                "name": "grid:astar",
                "diagonal_movement": "always",
                "weight": float(w),
                "cost_model": cost_model_name,
                "heuristic": "euclidean",
            }
            path = find_path(crop, start, end, solver_config)
            # print(f"Cost model: {cost_model_name}, weight: {w} => Path length: {len(path)}")
            results[cost_model_name][w] = path
    return results


def make_tensor_vis(
    crop: np.ndarray, start: tuple[int, int], end: tuple[int, int]
) -> tuple[np.ndarray, tuple[float, float]]:
    # Use raw elevation with provided range for consistent normalization in visualization
    elev_min = float(np.min(crop))
    elev_max = float(np.max(crop))
    marker_channel = np.zeros_like(crop, dtype=np.float32)
    marker_channel[start] = 1.0
    marker_channel[end] = 1.0
    obstacles_channel = np.zeros_like(crop, dtype=np.float32)
    tensor_vis = np.stack([crop.astype(np.float32), marker_channel, obstacles_channel], axis=-1)
    return tensor_vis, (elev_min, elev_max)


def hausdorff_distance(path1: list, path2: list) -> float:
    """Compute Hausdorff distance between two paths using scipy.directed_hausdorff."""
    if len(path1) == 0 or len(path2) == 0:
        return 0.0
    arr1 = np.array(path1)
    arr2 = np.array(path2)
    d1 = directed_hausdorff(arr1, arr2)[0]
    d2 = directed_hausdorff(arr2, arr1)[0]
    return max(d1, d2)

In [22]:
from tqdm import tqdm


samples = []
records = []

for i in tqdm(range(NUM_SAMPLES)):
    crop = generate_random_crop(RESOLUTION, tile_stats_df, TILES_DIR, global_min, global_max, seed=1000 + i)
    if crop is None:
        print(f"Sample {i + 1}: failed to generate crop")
        continue
    # print(f"Sample {i + 1}: generated crop with minimum elevation {np.min(crop):.1f} and maximum elevation {np.max(crop):.1f}")
    start, end = get_marker_positions(RESOLUTION)
    paths = compute_paths_for_crop(crop, start, end, COST_MODELS_LIST, WEIGHTS)
    samples.append({"index": i + 1, "crop": crop, "start": start, "end": end, "paths": paths})

    for cost_model_name in COST_MODELS_LIST:
        for w in WEIGHTS:
            path = paths[cost_model_name][w]
            records.append(
                {
                    "sample": i + 1,
                    "cost_model": cost_model_name,
                    "weight": w,
                    "path_length": len(path),
                    "found": len(path) > 0,
                }
            )


print(f"\nGenerated {len(samples)}/{NUM_SAMPLES} samples")

  0%|          | 0/1000 [00:00<?, ?it/s]

100%|██████████| 1000/1000 [23:37<00:00,  1.42s/it]


Generated 1000/1000 samples


In [23]:
results_df = pd.DataFrame(records)
results_df.head()

,sample,cost_model,weight,path_length,found
0,1,distance,1.000000,154,True
1,1,distance,1.511775,154,True
2,1,distance,2.285464,154,True
3,1,distance,3.455107,154,True
4,1,distance,5.223345,154,True


In [24]:
summary_df = (
    results_df.groupby(["cost_model", "weight"])
    .agg(
        mean_path_length=("path_length", "mean"),
        min_path_length=("path_length", "min"),
        max_path_length=("path_length", "max"),
        found_rate=("found", "mean"),
    )
    .reset_index()
)
summary_df

,cost_model,weight,mean_path_length,min_path_length,max_path_length,found_rate
0,distance,1.000000e+00,154.000,154,154,1.0
1,distance,1.511775e+00,154.000,154,154,1.0
2,distance,2.285464e+00,154.000,154,154,1.0
3,distance,3.455107e+00,154.000,154,154,1.0
4,distance,5.223345e+00,154.000,154,154,1.0
...,...,...,...,...,...,...
155,slope_uphill,1.914482e+06,258.801,154,586,1.0
156,slope_uphill,2.894266e+06,258.801,154,586,1.0
157,slope_uphill,4.375479e+06,258.801,154,586,1.0
158,slope_uphill,6.614741e+06,258.801,154,586,1.0


In [34]:
from matplotlib.lines import Line2D

mats = {w: list() for w in WEIGHTS}

# Per-sample grid of elevation + path visualizations (rows = cost models, cols = weights)
for isample, sample in tqdm(enumerate(samples), total=NUM_SAMPLES):
    crop = sample["crop"]
    start = sample["start"]
    end = sample["end"]
    paths = sample["paths"]

    # Compute and display Hausdorff distance matrices (for each weight)
    for w in WEIGHTS:
        n = len(COST_MODELS_LIST)
        mat = np.zeros((n, n), dtype=float)
        for i, cm1 in enumerate(COST_MODELS_LIST):
            for j, cm2 in enumerate(COST_MODELS_LIST):
                p1 = paths[cm1][w]
                p2 = paths[cm2][w]
                mat[i, j] = hausdorff_distance(p1, p2)
        mats[w].append(mat)

    if isample >= MAX_SAMPLES_DISPLAY:
        continue

    elev_min, elev_max = float(np.min(crop)), float(np.max(crop))

    rows = len(COST_MODELS_LIST)
    cols = len(WEIGHTS)
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
    if rows == 1:
        axes = axes.reshape(1, -1)
    if cols == 1:
        axes = axes.reshape(-1, 1)

    for r, cm in enumerate(COST_MODELS_LIST):
        for c, w in enumerate(WEIGHTS):
            ax = axes[r, c]
            last_im = ax.imshow(crop, cmap="terrain", vmin=elev_min, vmax=elev_max, origin="upper")
            path = paths[cm][w]
            if len(path) > 0:
                pa = np.array(path)
                ax.plot(pa[:, 1], pa[:, 0], color="red", linewidth=2)
            ax.plot(start[1], start[0], "o", color="lime", markersize=6)
            ax.plot(end[1], end[0], "x", color="cyan", markersize=6)
            ax.set_title(f"{cm} | w={w} | len={len(path)}", fontsize=9)
            ax.axis("off")

    # Figure-level legend
    legend_handles = [
        Line2D([0], [0], color="red", lw=2, label="Path"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="lime", markersize=6, label="Start"),
        Line2D([0], [0], marker="x", color="cyan", markersize=6, label="End"),
    ]
    fig.legend(handles=legend_handles, loc="upper right")
    fig.suptitle(f"Sample {sample['index']}")
    plt.tight_layout()
    plt.show()

100%|██████████| 1000/1000 [06:35<00:00,  2.53it/s]


In [35]:
for w, k in mats.items():
    mats[w] = np.mean(np.array(k), axis=0)

In [36]:
# for w in WEIGHTS:
#     mat = mats[w]
#     df_mat = pd.DataFrame(mat, index=COST_MODELS_LIST, columns=COST_MODELS_LIST)
#     print(f"\nHausdorff distance matrix (sample={sample['index']}, weight={w}):")
#     display(df_mat)

#     fig2, ax2 = plt.subplots(figsize=(6, 6))
#     im = ax2.imshow(df_mat, cmap="viridis", origin="upper")
#     ax2.set_xticks(range(n))
#     ax2.set_xticklabels(COST_MODELS_LIST, rotation=90)
#     ax2.set_yticks(range(n))
#     ax2.set_yticklabels(COST_MODELS_LIST)
#     ax2.set_title(f"Hausdorff distance matrix (w={w})")
#     fig2.colorbar(im, ax=ax2)
#     plt.tight_layout()
#     plt.show()

In [37]:
# n = len(COST_MODELS_LIST)
# fig, axes = plt.subplots(n, n, figsize=(4 * n, 4 * n), sharex=True, sharey=True)
# if n == 1:
#     axes = np.array([[axes]])
# for i in range(n):
#     for j in range(n):
#         ax = axes[i, j]
#         y = [mats[w][i, j] for w in WEIGHTS]
#         ax.plot(WEIGHTS, y, marker="o", lw=1)
#         ax.set_xscale("log")
#         ax.grid(True, linestyle=":", alpha=0.5)
#         ax.set_title(f"{COST_MODELS_LIST[i]} ⟷ {COST_MODELS_LIST[j]}", fontsize=8)
#         if i == n - 1:
#             ax.set_xticks(WEIGHTS)
#             ax.set_xticklabels([str(int(w)) for w in WEIGHTS], rotation=45, fontsize=7)
#         else:
#             ax.tick_params(labelbottom=False)
#         if j == 0:
#             ax.set_ylabel("Hausdorff", fontsize=8)
#     break

# fig.suptitle("Hausdorff distance vs weight for each cost-model pair", fontsize=12)
# plt.tight_layout(rect=[0, 0, 1, 0.96])
# plt.show()

In [61]:
n = len(COST_MODELS_LIST)
fig, axes = plt.subplots(figsize=(10, 10), sharex=True)
ax = axes
for j in range(n):
    if j == 0:
        continue
    y = [mats[w][0, j] for w in WEIGHTS]
    ax.plot(WEIGHTS, y, marker="o", lw=1, label=f"{COST_MODELS_LIST[i]} ⟷ {COST_MODELS_LIST[j]}")
    print(WEIGHTS[np.argmax(y)])
ax.set_xscale("log")
ax.set_title(f"Hausdorff distances from {COST_MODELS_LIST[i]}", fontsize=10)
ax.set_ylabel("Hausdorff", fontsize=8)
ax.legend(fontsize=8)
plt.xlabel("Weight")
plt.tight_layout()
plt.grid("on")
# mark maxima in red and annotate with the associated weight
i = 0
for j in range(1, n):
    y = np.array([mats[w][i, j] for w in WEIGHTS])
    idx = int(np.nanargmax(y))
    w_max = WEIGHTS[idx]
    y_max = y[idx]
    ax.scatter(w_max, y_max, color="red", s=40, zorder=10)
    ax.annotate(f"{w_max:.0f}", xy=(w_max, y_max), xytext=(5, 5), textcoords="offset points", fontsize=8, color="red")
    # mark first weight that reaches >=99% of the maximum (per curve)
for j in range(1, n):
    y = np.array([mats[w][i, j] for w in WEIGHTS])
    if np.all(np.isnan(y)):
        continue
    y_max = np.nanmax(y)
    thresh = 0.995 * y_max
    idxs = np.where(y >= thresh)[0]
    if idxs.size == 0:
        continue
    idx99 = int(idxs[0])
    w99 = WEIGHTS[idx99]
    y99 = y[idx99]
    ax.scatter(w99, y99, color="purple", s=40, zorder=12)
    ax.annotate(f"{w99:.0f}", xy=(w99, y99), xytext=(5, -10), textcoords="offset points", fontsize=8, color="purple")
plt.show()

30702.9062975785
2894266.1247167517
6614740.6412301455


In [62]:
weights = {
    "elevation": 2000,
    "energy": 4000,
    "slope_uphill": 100000,
}